In [0]:
%sql
SELECT * FROM catalog.silver.fact_visits;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.table("catalog.silver.fact_visits")

In [0]:
# Readmission analysis by hospital and diagnosis
gold_df = (
    df.groupBy(
        "hospital_id",
        "hospital_name",
        "diagnosis_desc",
    )
    .agg(
        count("*").alias("total_visits"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(sum(col("is_30day_readmission").cast("int")) / count("*"), 3).alias("readmission_rate"),
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php")).alias("avg_cost")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_df.orderBy("readmission_rate", ascending=False))

In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.hospital_diagnosis_kpi")

In [0]:
# Hospital Region KPI
gold_region_df = (
    df.groupBy(
        "network_region_cluster"
    )
    .agg(
        count("*").alias("total_visits"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(
            sum(col("is_30day_readmission").cast("int")) / count("*"),
            3
        ).alias("readmission_rate"),
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php"), 2).alias("avg_cost")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_region_df.orderBy("readmission_rate", ascending=False))

In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.hospital_region_kpi")

In [0]:
# Diagnosis KPI 
gold_diagnosis_df = (
    df.groupBy(
        "diagnosis_desc",
        "diagnosis_code",
        "diagnosis_category"
    )
    .agg(
        count("*").alias("total_cases"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(avg(col("is_30day_readmission").cast("int")), 3).alias("readmission_rate"),
        round(avg(col("total_cost_php")),2).alias("avg_cost"),
        round(avg(col("length_of_stay_days")), 2).alias("avg_length_of_stay")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_diagnosis_df.orderBy("readmission_rate", ascending=False))

In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.diagnosis_kpi")

In [0]:
# Hospitals KPI
gold_hospitals_df = (
    df.groupBy(
        "hospital_id",
        "hospital_name",
        "hospital_class"
    )
    .agg(
        count("*").alias("total_visits"),
        countDistinct("patient_id").alias("total_patients"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(avg(col("is_30day_readmission").cast("int")), 3).alias("readmission_rate"),
        round(avg(col("length_of_stay_days")), 2).alias("avg_length_of_stay"),
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php"), 2).alias("avg_cost_per_visit")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_hospitals_df.orderBy("readmission_rate", ascending=False))

In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.hospitals_kpi")

In [0]:
# Physicians KPI
gold_physicians_df = (
    df.groupBy(
        "attending_physician_id",
        "specialization"
    )
    .agg(
        count("*").alias("total_visits"),
        countDistinct("patient_id").alias("total_patients"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions"),
        round(avg(col("is_30day_readmission").cast("int")), 3).alias("readmission_rate"),
        round(avg(col("length_of_stay_days")), 2).alias("avg_length_of_stay"),
        round(avg("total_cost_php"), 2).alias("avg_cost_per_patient")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_physicians_df.orderBy("readmission_rate", ascending=False)
)

In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.physicians_kpi")

In [0]:
# Cost KPI
gold_cost_df = (
    df.groupBy(
        "hospital_id",
        "hospital_name",
        "diagnosis_desc"
    )
    .agg(
        sum("total_cost_php").alias("total_cost"),
        round(avg("total_cost_php"), 2).alias("avg_cost"),
        round(avg("length_of_stay_days"), 2).alias("avg_length_of_stay"),
        sum(col("is_30day_readmission").cast("int")).alias("total_readmissions")
    )
    .withColumn("gold_load_timestamp", current_timestamp()
    )
)

In [0]:
display(gold_cost_df)

In [0]:
gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("catalog.gold.cost_kpi")